# 使用单隐藏层神经网络进行平面数据分类

欢迎开始第 3 周的编程作业！本次将构建你的第一个神经网络，它包含一个隐藏层。你会看到，这个模型与此前使用逻辑回归实现的模型有明显差别。

读完本作业，你将能够：

- 使用单个隐藏层实现 2 类分类神经网络
- 使用具有非线性激活函数的单位，例如 tanh
- 计算交叉熵损失
- 实现前向传播与反向传播

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/neural-networks-deep-learning/supplement/xLEC1/optional-downloading-your-notebook-downloading-your-workspace-and-refreshing) 获取一份新的作业。

## 目录
- [1 - 依赖库](#1)
- [2 - 加载数据集](#2)
    - [练习 1](#ex-1)
- [3 - 简单逻辑回归](#3)
- [4 - 神经网络模型](#4)
    - [4.1 - 定义神经网络结构](#4-1)
        - [练习 2 - layer_sizes](#ex-2)
    - [4.2 - 初始化模型参数](#4-2)
        - [练习 3 - initialize_parameters](#ex-3)
    - [4.3 - 循环](#4-3)
        - [练习 4 - forward_propagation](#ex-4)
    - [4.4 - 计算代价](#4-4)
        - [练习 5 - compute_cost](#ex-5)
    - [4.5 - 实现反向传播](#4-5)
        - [练习 6 - backward_propagation](#ex-6)
    - [4.6 - 更新参数](#4-6)
        - [练习 7 - update_parameters](#ex-7)
    - [4.7 - 集成](#4-7)
        - [练习 8 - nn_model](#ex-8)
- [5 - 测试模型](#5)
    - [5.1 - 预测](#5-1)
        - [练习 9 - predict](#ex-9)
    - [5.2 - 在平面数据集上测试模型](#5-2)
- [6 - 调整隐藏层大小（选做，不计分）](#6)
- [7- 其他数据集上的性能](#7)

<a name='1'></a>
# 1 - 依赖库

首先导入作业期间需要的所有包。

- [NumPy](https://www.numpy.org/) 是使用 Python 进行科学计算的基础包。
- [sklearn](http://scikit-learn.org/stable/) 为数据挖掘和数据分析提供简单高效的工具。 
- [matplotlib](http://matplotlib.org) 是一个用 Python 绘制图形的库。
- testCases 提供了一些测试示例来评估你的函数的正确性
- planar_utils 提供了在此作业中使用的各种有用的函数

In [ ]:
### v2.0

In [ ]:
# 包导入
import numpy as np
import copy
import matplotlib.pyplot as plt
from testCases_v2 import *
from public_tests import *
import sklearn
import sklearn.datasets
import sklearn.linear_model
from planar_utils import plot_decision_boundary, sigmoid, load_planar_dataset, load_extra_datasets

%matplotlib inline

%load_ext autoreload
%autoreload 2

<a name='2'></a>
# 2 - 加载数据集 



In [ ]:
X, Y = load_planar_dataset()

使用 matplotlib 可视化数据集。数据看起来像一朵“花”，有一些红色（标签 y=0）和一些蓝色（y=1）点。你的目标是构建一个模型来适应这些数据。换句话说，我们希望分类器将区域定义为红色或蓝色。

In [ ]:
# 可视化数据：
plt.scatter(X[0, :], X[1, :], c=Y, s=40, cmap=plt.cm.Spectral);

你有：
    - 一个 NumPy 数组（矩阵）X，其中包含你的特征（x1，x2）
    - 包含标签的 NumPy 数组（向量）Y（红色：0，蓝色：1）。

首先，更好地了解你的数据是什么样的。 

<a name='ex-1'></a>
### 练习 1 

你有多少个训练样本？另外，变量`X`和`Y`的`shape`是多少？ 

**提示**：如何获得 NumPy 数组的形状？ [（帮助）](https://docs.scipy.org/doc/numpy/reference/generated/numpy.ndarray.shape.html)

In [ ]:
# （约3行代码）
# shape_X = ...
# shape_Y = ...
# 训练集尺寸
# m = ...
# YOUR CODE STARTS HERE


# YOUR CODE ENDS HERE

print ('The shape of X is: ' + str(shape_X))
print ('The shape of Y is: ' + str(shape_Y))
print ('I have m = %d training examples!' % (m))

**预期输出**：
       
<table style="width:20%">
  <tr>
    <td> X 的形状 </td>
    <td> (2, 400) </td> 
  </tr>
  <tr>
    <td>Y</td> 的形状
    <td>(1, 400) </td> 
    </tr>
    <tr>
    <td>m</td>
    <td> 400 </td> 
  </tr>
</table>

<a name='3'></a>
## 3 - 简单逻辑回归

在构建完整的神经网络之前，让我们检查一下逻辑回归在这个问题上的表现。你可以使用 sklearn 的内置函数来实现此目的。运行下面的代码以在数据集上训练逻辑回归分类器。

In [ ]:
# 训练逻辑回归分类器
clf = sklearn.linear_model.LogisticRegressionCV();
clf.fit(X.T, Y.T);

你现在可以绘制这些模型的决策边界！运行下面的代码。

In [ ]:
# 绘制逻辑回归的决策边界
plot_decision_boundary(lambda x: clf.predict(x), X, Y)
plt.title("Logistic Regression")

# 打印精度
LR_predictions = clf.predict(X.T)
print ('Accuracy of logistic regression: %d ' % float((np.dot(Y,LR_predictions) + np.dot(1-Y,1-LR_predictions))/float(Y.size)*100) +
       '% ' + "(percentage of correctly labelled datapoints)")

**预期输出**：

<table style="width:20%">
  <tr>
    <td>精度</td>
    <td> 47% </td> 
  </tr>
  
</table>


**解释**：数据集不是线性可分的，因此逻辑回归表现不佳。希望神经网络能做得更好。我们现在就试试这个吧！ 

<a name='4'></a>
## 4 - 神经网络模型

逻辑回归在花卉数据集上效果不佳。接下来，你将使用单个隐藏层训练神经网络，并了解它如何处理相同的问题。

**模型**：
<img src="images/classification_kiank.png" style="width:600px;height:300px;">

**数学上**：

例如 $x^{(i)}$：
$$z^{[1] (i)} =  W^{[1]} x^{(i)} + b^{[1]}\tag{1}$$ 
$$a^{[1] (i)} = \tanh(z^{[1] (i)})\tag{2}$$
$$z^{[2] (i)} = W^{[2]} a^{[1] (i)} + b^{[2]}\tag{3}$$
$$\hat{y}^{(i)} = a^{[2] (i)} = \sigma(z^{ [2] (i)})\tag{4}$$
$$y^{(i)}_{prediction} = \begin{cases} 1 & \mbox{if } a^{[2](i)} > 0.5 \\ 0 & \mbox{otherwise } \end{cases}\tag{5}$$

鉴于对所有示例的预测，你还可以计算代价 $J$，如下所示： 
$$J = - \frac{1}{m} \sum\limits_{i = 0}^{m} \large\left(\small y^{(i)}\log\left(a^{[2] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[2] (i)}\right)  \large  \right) \small \tag{6}$$

**提醒**：构建神经网络的一般方法是：
    1. 定义神经网络结构（输入单元数、隐藏单元数等）。 
    2. 初始化模型参数
    3. 循环：
        - 实现前向传播
        - 计算损失
        - 实现反向传播以获得梯度
        - 更新参数（梯度下降）

实际实现时，通常先编写辅助函数完成步骤 1–3，再将它们组合成 `nn_model()` 函数。构建好 `nn_model()` 并训练得到合适的参数后，就可以对新数据进行预测。

<a name='4-1'></a>
### 4.1 - 定义神经网络结构 ####

<a name='ex-2'></a>
### 练习 2 - layer_sizes 

定义三个变量：
- n_x：输入层的大小
- n_h：隐藏层的大小（**将其设置为 4，如 `n_h = 4`，但仅适用于本练习 2**） 
- n_y：输出层的大小

**提示**：使用X和Y的形状来找到n_x和n_y。另外，将隐藏层大小硬编码为 4。

In [ ]:
# GRADED FUNCTION: layer_sizes

def layer_sizes(X, Y):
    """
    参数：
    X -- 输入形状数据集（输入大小，示例数量）
    Y -- 形状标签（输出大小、示例数量）
    
    返回：
    n_x -- 输入层的大小
    n_h -- 隐藏层的大小
    n_y -- 输出层的大小
    """
    #（约3行代码）
    # n_x = ... 
    # n_h = ...
    # n_y = ... 
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    return (n_x, n_h, n_y)

In [ ]:
t_X, t_Y = layer_sizes_test_case()
(n_x, n_h, n_y) = layer_sizes(t_X, t_Y)
print("The size of the input layer is: n_x = " + str(n_x))
print("The size of the hidden layer is: n_h = " + str(n_h))
print("The size of the output layer is: n_y = " + str(n_y))

layer_sizes_test(layer_sizes)

***预期输出***
```
The size of the input layer is: n_x = 5
The size of the hidden layer is: n_h = 4
The size of the output layer is: n_y = 2
All tests passed!
```

<a name='4-2'></a>
### 4.2 - 初始化模型参数 ####

<a name='ex-3'></a>
### 练习 3 - initialize_parameters

实现函数 `initialize_parameters()`。

**说明**：
- 确保参数大小正确。如果需要的话可以参考上面的神经网络图。
- 你将使用随机值初始化权重矩阵。 
    - 使用：`np.random.randn(a,b) * 0.01` 随机初始化形状为 (a,b) 的矩阵。
- 你将把偏置向量初始化为零。 
    - 使用：`np.zeros((a,b))` 初始化形状为 (a,b) 且为零的矩阵。

In [ ]:
# GRADED FUNCTION: initialize_parameters

def initialize_parameters(n_x, n_h, n_y):
    """
    论证：
    n_x -- 输入层的大小
    n_h -- 隐藏层的尺寸
    n_y -- 输出层的大小
    
    返回：
    params -- 包含参数的 python 字典：
                    W1 -- 形状 (n_h, n_x) 的权重矩阵
                    b1 -- 形状为 (n_h, 1) 的偏置向量
                    W2 -- 形状 (n_y, n_h) 的权重矩阵
                    b2 -- 形状为 (n_y, 1) 的偏置向量
    """    
    #（约4行代码）
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE

    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters

In [ ]:
np.random.seed(2)
n_x, n_h, n_y = initialize_parameters_test_case()
parameters = initialize_parameters(n_x, n_h, n_y)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test(initialize_parameters)

**预期输出**
```
W1 = [[-0.00416758 -0.00056267]
 [-0.02136196  0.01640271]
 [-0.01793436 -0.00841747]
 [ 0.00502881 -0.01245288]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01057952 -0.00909008  0.00551454  0.02292208]]
b2 = [[0.]]
All tests passed!
```

<a name='4-3'></a>
### 4.3 - 循环 

<a name='ex-4'></a>
### 练习 4 - forward_propagation

使用以下等式实现 `forward_propagation()`：

$$Z^{[1]} =  W^{[1]} X + b^{[1]}\tag{1}$$ 
$$A^{[1]} = \tanh(Z^{[1]})\tag{2}$$
$$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}\tag{3}$$
$$\hat{Y} = A^{[2]} = \sigma(Z^{[2]})\tag{4}$$


**说明**：

- 检查上图中分类器的数学表示。
- 使用函数 `sigmoid()`。它内置于（进口）Notebook 中。
- 使用函数 `np.tanh()`。它是 NumPy 库的一部分。
- 使用以下步骤实现：
    1. 从字典“parameters”中检索每个参数（这是使用 `parameters[".."]` 得到的 `initialize_parameters()` 的输出）。
    2. 实现前向传播。计算 $Z^{[1]}, A^{[1]}, Z^{[2]}$ 和 $A^{[2]}$（对训练集中所有示例的所有预测的向量）。
- 反向传播中所需的值存储在“缓存”中。缓存将作为反向传播函数的输入给出。

In [ ]:
# GRADED FUNCTION:forward_propagation

def forward_propagation(X, parameters):
    """
    论证：
    X -- 输入大小为 (n_x, m) 的数据
    parameters -- 包含参数的python字典（初始化函数的输出）
    
    返回：
    A2 -- 第二个激活值的 sigmoid 输出
    cache -- 包含 "Z1"、"A1"、"Z2" 和 "A2" 的字典
    """
    # 从字典 "parameters" 中检索每个参数
    #（约4行代码）
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 实现前向传播来计算A2（概率）
    # （约4行代码）
    # Z1 = ...
    # A1 = ...
    # Z2 = ...
    # A2 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    assert(A2.shape == (1, X.shape[1]))
    
    cache = {"Z1": Z1,
             "A1": A1,
             "Z2": Z2,
             "A2": A2}
    
    return A2, cache

In [ ]:
t_X, parameters = forward_propagation_test_case()
A2, cache = forward_propagation(t_X, parameters)
print("A2 = " + str(A2))

forward_propagation_test(forward_propagation)

***预期输出***
```
A2 = [[0.21292656 0.21274673 0.21295976]]
All tests passed!
```

<a name='4-4'></a>
### 4.4 - 计算代价

现在你已经计算了 $A^{[2]}$（在 Python 变量“`A2`”中），其中所有示例都包含 $a^{[2](i)}$，你可以按如下方式计算代价函数：

$$J = - \frac{1}{m} \sum\limits_{i = 1}^{m} \large{(} \small y^{(i)}\log\left(a^{[2] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[2] (i)}\right) \large{)} \small\tag{13}$$

<a name='ex-5'></a>
### 练习 5 - compute_cost 

实现 `compute_cost()` 来计算代价 $J$ 的值。

**说明**：
- 实现交叉熵损失的方法有很多种。这是在不使用 for 循环的情况下实现方程的一部分的一种方法：
$- \sum\limits_{i=1}^{m}  y^{(i)}\log(a^{[2](i)})$:
```python
logprobs = np.multiply(np.log(A2),Y)
cost = - np.sum(logprobs)          
```

- 用它来构建代价函数的整个表达。

**注释**： 

- 你可以先使用 `np.multiply()`，然后再使用 `np.sum()`，或者直接使用 `np.dot()`）。  
- 如果你使用 `np.multiply` 后跟 `np.sum`，则最终结果将是 `float` 类型，而如果你使用 `np.dot`，则结果将是 2D NumPy 数组。  
- 你可以使用 `np.squeeze()` 删除冗余维度（在单浮点数的情况下，这将减少为零维数组）。 
- 你还可以使用 `float()` 将数组转换为类型 `float`。

In [ ]:
# GRADED FUNCTION: compute_cost

def compute_cost(A2, Y):
    """
    计算等式（13）中给出的交叉熵代价
    
    参数：
    A2 -- 第二个激活值的 sigmoid 输出，形状（1，示例数量）
    Y -- "true" 标签形状向量（1，示例数量）

    返回：
    cost -- 给定方程（13）的交叉熵代价
    
    """
    
    m = Y.shape[1] # 示例数量

    # 计算交叉熵代价
    # （约2行代码）
    # logprobs = ...
    # cost = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    cost = float(np.squeeze(cost))  # 确保代价符合我们的预期。 
                                    # E.g.，将 [[17]] 变为 17 
    
    return cost

In [ ]:
A2, t_Y = compute_cost_test_case()
cost = compute_cost(A2, t_Y)
print("cost = " + str(compute_cost(A2, t_Y)))

compute_cost_test(compute_cost)

***预期输出***
```
cost = 0.6930587610394646
All tests passed!
``` 


<a name='4-5'></a>
### 4.5 - 实现反向传播

使用前向传播期间计算的缓存，你现在可以实现反向传播。

<a name='ex-6'></a>
### 练习 6 - backward_propagation

实现函数 `backward_propagation()`。

**说明**：
反向传播通常是深度学习中最难（最数学化）的部分。为了帮助你，这里再次提供反向传播讲座中的幻灯片。你将需要使用本幻灯片右侧的六个方程，因为你正在构建向量化的实现。  

<img src="images/grad_summary.png" style="width:600px;height:300px;">
<caption><center><font color='purple'><b>图1</b>：反向传播。使用右边的六个方程。</font></center></caption>

<!--
$\frac{\partial \mathcal{J} }{ \partial z_{2}^{(i)} } = \frac{1}{m} (a^{[2](i)} - y^{(i)})$

$\frac{\partial \mathcal{J} }{ \partial W_2 } = \frac{\partial \mathcal{J} }{ \partial z_{2}^{(i)} } a^{[1] (i) T} $

$\frac{\partial \mathcal{J} }{ \partial b_2 } = \sum_i{\frac{\partial \mathcal{J} }{ \partial z_{2}^{(i)}}}$

$\frac{\partial \mathcal{J} }{ \partial z_{1}^{(i)} } =  W_2^T \frac{\partial \mathcal{J} }{ \partial z_{2}^{(i)} } * ( 1 - a^{[1] (i) 2}) $

$\frac{\partial \mathcal{J} }{ \partial W_1 } = \frac{\partial \mathcal{J} }{ \partial z_{1}^{(i)} }  X^T $

$\frac{\partial \mathcal{J} _i }{ \partial b_1 } = \sum_i{\frac{\partial \mathcal{J} }{ \partial z_{1}^{(i)}}}$

- 请注意，$*$ 表示元素乘法。
- 你将使用的符号在深度学习编码中很常见：
    - dW1 = $\frac{\partial \mathcal{J} }{ \partial W_1 }$
    - db1 = $\frac{\partial \mathcal{J} }{ \partial b_1 }$
    - dW2 = $\frac{\partial \mathcal{J} }{ \partial W_2 }$
    - db2 = $\frac{\partial \mathcal{J} }{ \partial b_2 }$
    
!-->

- 温馨提示：
    - 要计算 dZ1，你需要计算 $g^{[1]'}(Z^{[1]})$。由于 $g^{[1]}(.)$ 是 tanh 激活函数，如果 $a = g^{[1]}(z)$ 则为 $g^{[1]'}(z) = 1-a^2$。所以你可以计算 
    $g^{[1]'}(Z^{[1]})$ 使用 `(1 - np.power(A1, 2))`。

In [ ]:
# GRADED FUNCTION: backward_propagation

def backward_propagation(parameters, cache, X, Y):
    """
    使用上述说明实现反向传播。
    
    参数：
    parameters -- 包含我们参数的Python字典 
    cache -- 包含 "Z1"、"A1"、"Z2" 和 "A2" 的字典。
    X -- 形状的输入数据（2，示例数）
    Y -- "true" 标签形状向量（1，示例数量）
    
    返回：
    grads -- 包含不同参数的梯度的 python 字典
    """
    m = X.shape[1]
    
    # 首先，从字典 "parameters" 中检索 W1 和 W2。
    #（约2行代码）
    # W1 = ...
    # W2 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
        
    # 从字典 "cache" 中检索 A1 和 A2。
    #（约2行代码）
    # A1 = ...
    # A2 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 反向传播：计算dW1、db1、dW2、db2。 
    #（约6行代码，对应上面幻灯片中的6个方程）
    # dZ2 = ...
    # dW2 = ...
    # db2 = ...
    # dZ1 = ...
    # dW1 = ...
    # db1 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    grads = {"dW1": dW1,
             "db1": db1,
             "dW2": dW2,
             "db2": db2}
    
    return grads

In [ ]:
parameters, cache, t_X, t_Y = backward_propagation_test_case()

grads = backward_propagation(parameters, cache, t_X, t_Y)
print ("dW1 = "+ str(grads["dW1"]))
print ("db1 = "+ str(grads["db1"]))
print ("dW2 = "+ str(grads["dW2"]))
print ("db2 = "+ str(grads["db2"]))

backward_propagation_test(backward_propagation)

***预期输出***
```
dW1 = [[ 0.00301023 -0.00747267]
 [ 0.00257968 -0.00641288]
 [-0.00156892  0.003893  ]
 [-0.00652037  0.01618243]]
db1 = [[ 0.00176201]
 [ 0.00150995]
 [-0.00091736]
 [-0.00381422]]
dW2 = [[ 0.00078841  0.01765429 -0.00084166 -0.01022527]]
db2 = [[-0.16655712]]
All tests passed!
```

<a name='4-6'></a>
### 4.6 - 更新参数 

<a name='ex-7'></a>
### 练习 7 - update_parameters

实现更新规则。使用梯度下降。你必须使用（dW1，db1，dW2，db2）才能更新（W1，b1，W2，b2）。

**一般梯度下降规则**：$\theta = \theta - \alpha \frac{\partial J }{ \partial \theta }$，其中$\alpha$是学习率，$\theta$代表参数。

<img src="images/sgd.gif" style="width:400;height:400;"> <img src="images/sgd_bad.gif" style="width:400;height:400;">
<caption><center><font color='purple'><b>图2</b>：梯度下降算法具有良好的学习率（收敛）和糟糕的学习率 （发散）。图片由 Adam Harley 提供。</font></center></caption>

**提示**

- 复制作为参数传递给函数的列表或字典时，请使用 `copy.deepcopy(...)`。它避免了在函数内修改输入参数。在某些情况下，这可能效率低下，但出于评分目的，这是必需的。


In [ ]:
# GRADED FUNCTION: update_parameters

def update_parameters(parameters, grads, learning_rate = 1.2):
    """
    使用上面给出的梯度下降更新规则更新参数
    
    参数：
    parameters -- 包含参数的Python字典 
    grads -- 包含你的梯度的 python 字典 
    
    返回：
    parameters -- 包含更新参数的Python字典 
    """
    # 从字典 "parameters" 中检索每个参数的副本。 W1 和 W2 使用 copy.deepcopy(...)
    #（约4行代码）
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 从字典 "grads" 中检索每个梯度
    #（约4行代码）
    # dW1 = ...
    # db1 = ...
    # dW2 = ...
    # db2 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 每个参数的更新规则
    #（约4行代码）
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters

In [ ]:
parameters, grads = update_parameters_test_case()
parameters = update_parameters(parameters, grads)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

update_parameters_test(update_parameters)

***预期输出***
```
W1 = [[-0.00643025  0.01936718]
 [-0.02410458  0.03978052]
 [-0.01653973 -0.02096177]
 [ 0.01046864 -0.05990141]]
b1 = [[-1.02420756e-06]
 [ 1.27373948e-05]
 [ 8.32996807e-07]
 [-3.20136836e-06]]
W2 = [[-0.01041081 -0.04463285  0.01758031  0.04747113]]
b2 = [[0.00010457]]
All tests passed!
```

<a name='4-7'></a>
### 4.7 - 集成

将你的函数集成到 `nn_model()` 中 

<a name='ex-8'></a>
### 练习 8 - nn_model

在 `nn_model()` 中构建神经网络模型。

**说明**：神经网络模型必须以正确的顺序使用之前的函数。

In [ ]:
# GRADED FUNCTION: nn_model

def nn_model(X, Y, n_h, num_iterations = 10000, print_cost=False):
    """
    参数：
    X -- 形状数据集（2，示例数量）
    Y -- 形状标签（1，示例数量）
    n_h -- 隐藏层的尺寸
    num_iterations -- 梯度下降循环中的迭代次数
    print_cost -- 如果为 True，则每 1000 次迭代打印代价
    
    返回：
    parameters -- 模型学习到的参数。然后可以使用它们进行预测。
    """
    
    np.random.seed(3)
    n_x = layer_sizes(X, Y)[0]
    n_y = layer_sizes(X, Y)[2]
    
    # 初始化参数
    #（≈ 1 行代码）
    # parameters = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 循环 (梯度下降)

    for i in range(0, num_iterations):
         
        #（约4行代码）
        # 前向传播。输入："X, parameters"。输出："A2, cache"。
        # A2, cache = ...
        
        # 代价函数。输入："A2, Y"。输出："cost"。
        # cost = ...
 
        # 反向传播。输入："parameters, cache, X, Y"。输出："grads"。
        # grads = ...
 
        # 梯度下降参数更新。输入："parameters, grads"。输出："parameters"。
        # parameters = ...
        
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        # 打印每 1000 次迭代的代价
        if print_cost and i % 1000 == 0:
            print ("Cost after iteration %i: %f" %(i, cost))

    return parameters

In [ ]:
nn_model_test(nn_model)

***预期输出***
```
Cost after iteration 0: 0.693497
Cost after iteration 1000: 0.000180
Cost after iteration 2000: 0.000088
...
Cost after iteration 8000: 0.000022
Cost after iteration 9000: 0.000019
W1 = [[-0.48394893 -0.91443482]
 [-0.69227123 -1.28596651]
 [ 0.63806018  1.18124948]
 [ 0.73594679  1.35718308]
 [-0.62621054 -1.16022636]]
b1 = [[ 0.00285386]
 [ 0.01642488]
 [-0.01114986]
 [-0.01656405]
 [ 0.01042274]]
W2 = [[-1.45325879 -2.65768451  2.26286606  2.97274518 -2.18860534]]
b2 = [[0.00758617]]
All tests passed!
```

<a name='5'></a>
## 5 - 测试模型

<a name='5-1'></a>
### 5.1 - 预测

<a name='ex-9'></a>
### 练习 9 - predict

通过构建 `predict()` 使用模型进行预测。
使用前向传播来预测结果。

**提醒**：预测 = $y_{prediction} = \mathbb 1 \text{{activation > 0.5}} = \begin{cases}
      1 & \text{if}\ activation > 0.5 \\
      0 & \text{otherwise}
    \end{cases}$  
    
例如，如果你想根据阈值将矩阵 X 的条目设置为 0 和 1，你可以这样做：```X_new = (X > threshold)```

In [ ]:
# GRADED FUNCTION: predict

def predict(parameters, X):
    """
    使用学习到的参数，预测 X 中每个示例的类别
    
    参数：
    parameters -- 包含参数的Python字典 
    X -- 输入大小为 (n_x, m) 的数据
    
    返回
    predictions -- 我们模型的预测向量（红色：0 / 蓝色：1）
    """
    
    # 使用前向传播计算概率，并使用 0.5 作为阈值分类为 0/1。
    #（约2行代码）
    # A2, cache = ...
    # predictions = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return predictions

In [ ]:
parameters, t_X = predict_test_case()

predictions = predict(parameters, t_X)
print("Predictions: " + str(predictions))

predict_test(predict)

***预期输出***
```
Predictions: [[ True False  True]]
All tests passed!
```

<a name='5-2'></a>
### 5.2 - 在平面数据集上测试模型

是时候运行模型并查看它在平面数据集上的表现了。运行以下代码，使用 $n_h$ 隐藏单元的单个隐藏层测试你的模型！

In [ ]:
# 使用 n_h 维隐藏层构建模型
parameters = nn_model(X, Y, n_h = 4, num_iterations = 10000, print_cost=True)

# 绘制决策边界
plot_decision_boundary(lambda x: predict(parameters, x.T), X, Y)
plt.title("Decision Boundary for hidden layer size " + str(4))

In [ ]:
# 打印精度
predictions = predict(parameters, X)
print ('Accuracy: %d' % float((np.dot(Y, predictions.T) + np.dot(1 - Y, 1 - predictions.T)) / float(Y.size) * 100) + '%')

**预期输出**： 

<table style="width:30%">
  <tr>
    <td><b>精度</b></td>
    <td> 90% </td> 
  </tr>
</table>

与逻辑回归相比，准确性非常高。模型已经学会了花瓣的图案！与逻辑回归不同，神经网络甚至能够学习高度非线性的决策边界。 

### 恭喜你完成本次编程作业！ 

以下是你刚刚完成的所有工作的快速回顾： 

- 使用隐藏层构建了完整的 2 类分类神经网络
- 充分利用非线性单元
- 计算交叉熵损失
- 已实现前向传播与反向传播
- 看到了改变隐藏层大小的影响，包括过拟合。

你已经创建了一个可以学习模式的神经网络！佳作。下面有一些可选练习来尝试其他一些隐藏层大小和其他数据集。 

<a name='6'></a>
## 6 - 调整隐藏层大小（选做，不计分）

运行以下code(可能需要 1-2 分钟)。然后，观察不同隐藏层尺寸的模型的不同行为。

In [ ]:
# 这可能需要大约 2 分钟才能运行

plt.figure(figsize=(16, 32))
hidden_layer_sizes = [1, 2, 3, 4, 5]

# 你可以尝试不同的隐藏层尺寸
# 但请确保在提交作业之前将其设置为 "hidden_layer_sizes = [1, 2, 3, 4, 5]"
# hidden_layer_sizes = [1, 2, 3, 4, 5, 20, 50]

for i, n_h in enumerate(hidden_layer_sizes):
    plt.subplot(5, 2, i+1)
    plt.title('Hidden Layer of size %d' % n_h)
    parameters = nn_model(X, Y, n_h, num_iterations = 5000)
    plot_decision_boundary(lambda x: predict(parameters, x.T), X, Y)
    predictions = predict(parameters, X)
    accuracy = float((np.dot(Y,predictions.T) + np.dot(1 - Y, 1 - predictions.T)) / float(Y.size)*100)
    print ("Accuracy for {} hidden units: {} %".format(n_h, accuracy))

**解释**：
- 较大的模型（具有更多隐藏单元）能够更好地拟合训练集，直到最终最大的模型过拟合数据。 
- 最佳的隐藏层大小似乎在 n_h = 5 左右。事实上，这里的值似乎很好地拟合了数据，而不会引起明显的过拟合。
- 稍后，你将熟悉正则化，它允许你使用非常大的模型（例如 n_h = 50）而不会出现太多过拟合。 

**注意**：请记住通过单击右上角的蓝色“提交作业”按钮来提交作业。 

**如果你愿意，你可以探索一些可选/不计分的问题**： 
- 将 tanh 激活值更改为 sigmoid 激活函数或 ReLU 激活函数时会发生什么？
- 与 learning_rate 一起玩。会发生什么？
- 如果我们更改数据集怎么办？ （请参阅下面的第 7 部分！）

<a name='7'></a>
## 7- 其他数据集上的性能

如果需要，你可以为以下每个数据集重新运行整个 Notebook（减去数据集部分）。

In [ ]:
# 数据集
noisy_circles, noisy_moons, blobs, gaussian_quantiles, no_structure = load_extra_datasets()

datasets = {"noisy_circles": noisy_circles,
            "noisy_moons": noisy_moons,
            "blobs": blobs,
            "gaussian_quantiles": gaussian_quantiles}

### START CODE HERE ### (choose your dataset)
dataset = "noisy_moons"
### END CODE HERE ###

X, Y = datasets[dataset]
X, Y = X.T, Y.reshape(1, Y.shape[0])

# 使 blob 成为二进制
if dataset == "blobs":
    Y = Y%2

# 可视化数据
plt.scatter(X[0, :], X[1, :], c=Y, s=40, cmap=plt.cm.Spectral);

**参考文献**：

- http://cs231n.github.io/neural-networks-case-study/